# Module 9 practice – ML on ShopKart

**Video 9.2–9.6 (practice)** · ShopKart project

**Goal:** Practise the full workflow on new questions. 5 tasks.

**How to open in Colab:** go to colab.research.google.com → File → Upload notebook → choose this file. Then click the folder icon on the left and drag these data files in: `shopkart_clean.csv`. Then run cells from top to bottom (Shift + Enter).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix, precision_score, recall_score
df = pd.read_csv("shopkart_clean.csv")

### Task 1 – split
Create y = 1 if `payment_mode` is "COD", else 0. Use features category, city, unit_price, membership, age (one-hot encoded). Split 75/25 with random_state=42 and stratify. Print the COD share in train and test.

In [2]:
y_cod = (df["payment_mode"] == "COD").astype(int)
X_cod = pd.get_dummies(df[["category", "city", "unit_price", "membership", "age"]], drop_first=True, dtype=int)
Xa, Xb, ya, yb = train_test_split(X_cod, y_cod, test_size=0.25, random_state=42, stratify=y_cod)
print("COD share train:", round(ya.mean() * 100, 1), "% | test:", round(yb.mean() * 100, 1), "%")

COD share train: 19.7 % | test: 19.7 %


### Task 2 – regression
Predict **profit** of an order using category, quantity, discount, membership and city (no unit_price, no cost_price, no revenue). Report MAE (₹) and R² on the test set (test_size=0.2, random_state=42). Is it good?

In [3]:
Xp = pd.get_dummies(df[["category", "quantity", "discount", "membership", "city"]], drop_first=True, dtype=int)
yp = df["profit"]
a, b, c, d = train_test_split(Xp, yp, test_size=0.2, random_state=42)
m = LinearRegression().fit(a, c)
pp = m.predict(b)
print("MAE ₹", round(mean_absolute_error(d, pp)), "| R²", round(r2_score(d, pp), 2))
print("Baseline MAE ₹", round(mean_absolute_error(d, np.full(len(d), c.mean()))))
print("Interpretation: modest, like the revenue model – without the product price, profit is hard to predict.")

MAE ₹ 520 | R² 0.17
Baseline MAE ₹ 567
Interpretation: modest, like the revenue model – without the product price, profit is hard to predict.


### Task 3 – leakage spotting
For the target `returned`, which of these columns would be data leakage: `delivery_days`, `rating`, `discount`, `profit`, `order_month`? Write your answer as a markdown note, then check by training a depth-4 tree with and without `rating`.

In [4]:
y = (df["returned"] == "Y").astype(int)
base = pd.get_dummies(df[["category", "city", "payment_mode", "delivery_days", "discount", "membership", "age", "unit_price"]],
                      drop_first=True, dtype=int)
for name, X in [("without rating", base), ("with rating", base.assign(rating=df["rating"].fillna(df["rating"].median())))]:
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    t = DecisionTreeClassifier(max_depth=4, random_state=42).fit(a, c)
    print(name, "-> recall for returns:", round(recall_score(d, t.predict(b)), 2))
print("Answer: rating is leakage (given after the return decision). delivery_days, discount and order_month are known by shipping time. profit is not leakage for returns in this dataset, but it is just revenue minus cost - no new information.")

without rating -> recall for returns: 0.02
with rating -> recall for returns: 0.31
Answer: rating is leakage (given after the return decision). delivery_days, discount and order_month are known by shipping time. profit is not leakage for returns in this dataset, but it is just revenue minus cost - no new information.


### Task 4 – threshold
Train the logistic regression from Video 9.5 (same features, test_size=0.2, random_state=42, stratify=y). At threshold **0.25**, print the confusion matrix, precision and recall.

In [5]:
a, b, c, d = train_test_split(base, y, test_size=0.2, random_state=42, stratify=y)
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(a, c)
p = (lr.predict_proba(b)[:, 1] >= 0.25).astype(int)
print(confusion_matrix(d, p))
print("precision", round(precision_score(d, p), 2), "| recall", round(recall_score(d, p), 2))

[[807  56]
 [111  26]]
precision 0.32 | recall 0.19


### Task 5 – overfitting
Using the same split as Task 4, find the `max_depth` between 1 and 15 with the highest **test** accuracy, and the gap between train and test accuracy at depth 15.

In [6]:
scores = {dep: DecisionTreeClassifier(max_depth=dep, random_state=42).fit(a, c) for dep in range(1, 16)}
test_acc = {dep: m.score(b, d) for dep, m in scores.items()}
best = max(test_acc, key=test_acc.get)
print("Best depth:", best, "test accuracy", round(test_acc[best], 3))
print("Depth 15 gap (train - test):", round(scores[15].score(a, c) - scores[15].score(b, d), 3))

Best depth: 1 test accuracy 0.863
Depth 15 gap (train - test): 0.143
